# README

This notebook contains the code to reproduce the analysis and figures presented in the research paper:

**Kelly, C.J., Wilson, M., Warren, L.M. _et al._ Diagnostic accuracy, fairness and clinical implementation of AI for breast cancer screening: results of multicenter retrospective and prospective technical feasibility studies. _Nat Cancer_ (2026). https://doi.org/10.1038/s43018-026-01127-0**

## Overview

This notebook demonstrates how to generate the key performance metrics and visualizations reported in the study, which evaluated Google's mammography AI system (v1.2) across retrospective and prospective cohorts in the UK National Health Service (NHS).

The code includes plotting functions for:

* **Main Performance (Figure 2):** Generating Receiver Operating Characteristic (ROC) curves and calculating sensitivity and specificity at the case, breast, and lesion levels.
* **Subgroup Analysis (Figure 3):** assessing fairness and bias by breaking down performance across key subgroups, including age, ethnicity, breast density, and Index of Multiple Deprivation (IMD).
* **Prospective Monitoring (Figure 4):** Visualizing Cancer Detection Rate (CDR) vs. Recall Rate (RR) and monitoring performance stability over time during the prospective deployment.

## Disclaimer

The content of this research code repository (i) is not intended to be a medical device; and (ii) is not intended for clinical use of any kind, including but not limited to diagnosis or prognosis.

## License

See https://github.com/Google-Health/google-health/blob/master/LICENSE for details.

In [ ]:
import dataclasses
from datetime import datetime
import io
import itertools
from PIL import Image
import matplotlib.colors as colors
import matplotlib.lines as lines
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import matplotlib.legend as plt_legend
import pandas as pd
import numpy as np

In [ ]:
# Load Source Data Excel file published with the paper
# Please ensure that 'SourceData.xlsx' is in the same directory as this script, and that the filename matches exactly.
source_data_path = 'SourceData.xlsx'
all_sheets = pd.read_excel(source_data_path, sheet_name=None)

In [ ]:
display_subgroups = {
  'All': ['All Cases'],
  'By Screening Service': [f'Screening Service {i}' for i in range(1, 6)],
  'By Screening Service and Screen Type': list(itertools.chain(
      *[[f'{site} - Prevalent Screens',
         f'{site} - Incident Screens',]
        for site in [f'Screening Service {i}' for i in range(1, 6)]])),
  'By Screen Type': ['Prevalent Screens',
                     'Incident Screens',
                    ],
  'By Age': ['Age 50-54',
            'Age 55-59',
            'Age 60-64',
            'Age 65+',
            ],
  'By Index of Multiple Deprivation': [
      '1st',
      '2nd',
      '3rd',
      '4th',
      '5th',
      '6th',
      '7th',
      '8th',
      '9th',
      '10th',
      ],
  'By Ethnicity': [
      'Unknown',
      'White',
      'Asian',
      'Black',
      'Other',
      'Mixed',
  ],
  'By Scanner Model': [
      'Hologic',
      'HologicLoradSelenia',
      'HologicSeleniaDimensions',
      'Siemens',
      'GE',
  ],
  'By Breast Density': [
      'BIRADS-1',
      'BIRADS-2',
      'BIRADS-3',
      'BIRADS-4',
  ],
  'By Cancer Time-Point': [
      'Current Screen',
      'Interval',
      'Next Screen',
  ],
  'By Lesion Risk': [
      'High',
      'Low',
  ],
  'By Lesion Type': [
      'Invasive',
      'Invasive (<15mm)',
      'Invasive (>=15mm)',
      'Insitu',
  ],
  'By Lesion Size': [
      'Invasive (<15mm)',
      'Invasive (>=15mm)',
  ],
  'By Granular Lesion Size': [
      '<5mm',
      '5-9mm',
      '10-14mm',
      '15-19mm',
      '20-24mm',
      '25-29mm',
      '30-34mm',
      '35-39mm',
      '40mm+',

  ],
  'By Lesion Subtype': [
      'Unknown',
      'Invasive-G1',
      'Invasive-G2',
      'Invasive-G3',
      'Insitu-NDH',
      'Insitu-NDI',
      'Insitu-NDL',
  ],
}

In [ ]:
def marker_kwawgs(reader, color, zoomed=False, markersize=8, zoomed_markersize=10, markeredgewidth=0.5):
      return {
          'ai_read': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'s',
              'markeredgecolor': 'black',
              'markeredgewidth': markeredgewidth,
              'c': color,},
          'ai_read_strict': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'s',
              'markeredgecolor': 'black',
              'markeredgewidth': markeredgewidth,
              'c': color,},
          'ai_read_op1': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'s',
              'markeredgecolor': 'black',
              'markeredgewidth': markeredgewidth,
              'c': color,},
          'ai_read_op2': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'s',
              'markeredgecolor': 'black',
              'markeredgewidth': markeredgewidth,
              'c': color,},
          'human_first_read': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'o',
              'c': color,
              'markeredgecolor': color,
              'markerfacecolor': 'white',},
          'human_second_read': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'o',
              'c': color,
              'markeredgecolor': color,
              'markerfacecolor': color,
              'markerfacecoloralt': 'white',
              'fillstyle': 'right',},
          'human_consensus_read': {
              'markersize': zoomed_markersize if zoomed else markersize,
              'marker':'o',
              'markeredgecolor': 'black',
              'markeredgewidth': markeredgewidth,
              'c': color,},
      }.get(reader, {'marker': 'X'})

# 1 inch = 25.4 millimeters
mm_to_inch = 1 / 25.4

def to_inch(x):
  return x * mm_to_inch

In [ ]:
def plot_readers_and_auc_pair(ax1, ax2, **kwargs):
  plot_readers_and_auc(ax1, zoomed=False, **kwargs)
  plot_readers_and_auc(ax2, zoomed=True, **kwargs)

def plot_readers_and_auc(ax,
                         subgroups=None,
                         target=None,
                         zoom_xy = ([0.0, .1], [0.3, .65]),
                         readers = ['ai_read', 'human_first_read', 'human_second_read', 'human_consensus_read'],
                         zoomed=False,
                         subgroup_colors=colors.TABLEAU_COLORS,
                         supgroup_alphas=None,
                         minimum_n=None,
                         label_with_target=False,
                         roc_df=None,
                         marker_sizes=(8, 10),
):
  readers_props = {
      'ai_read': ('AI Read'),
      'ai_read_op1': ('AI Read (OP1)'),
      'ai_read_op2': ('AI Read (OP2)'),
      'human_first_read':  ('1st Human Read'),
      'human_second_read':  ('2nd Human Read'),
      'human_consensus_read': ('Human Consensus'),
  }
  if not zoomed:
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    if zoom_xy:
      ax.add_patch(patches.Rectangle(
          (zoom_xy[0][0], zoom_xy[1][0]),
          zoom_xy[0][1] - zoom_xy[0][0],
          zoom_xy[1][1] - zoom_xy[1][0],
          linewidth=1, edgecolor='grey', facecolor='none'))
  elif zoom_xy:
    ax.set_xlim(zoom_xy[0])
    ax.set_ylim(zoom_xy[1])

  if type(subgroups) == str:
    subgroups = [subgroups]
  if subgroups is None:
    subgroups = ['ALL']

  if supgroup_alphas is None:
    supgroup_alphas = [1.0] * len(subgroups)

  if roc_df is None:
    raise ValueError('roc_df is required')

  roc_rows = roc_df.set_index('Level')
  fpr_cols = sorted([c for c in roc_df.columns if c.startswith('fpr_')], key=lambda c: int(c.split('_')[1]))
  tpr_cols = sorted([c for c in roc_df.columns if c.startswith('tpr_')], key=lambda c: int(c.split('_')[1]))

  for i, (subgroup, color, alpha) in enumerate(zip(subgroups, list(subgroup_colors), supgroup_alphas)):
    if target not in roc_rows.index:
      continue
    roc_row = roc_rows.loc[target]
    if isinstance(roc_row, pd.DataFrame):
      roc_row = roc_row.iloc[0]
    fpr = roc_row[fpr_cols].to_numpy()
    tpr = roc_row[tpr_cols].to_numpy()
    auc = roc_row.get('AUC')

    if minimum_n is not None and minimum_n > 0:
      continue

    auc_label = f'ROC [AUC={auc:0.2f}]' if pd.notna(auc) else 'ROC'
    handles = []
    labels = []
    handles.extend(ax.plot(fpr, tpr,
            zorder=-i,
            c=color,
            alpha=alpha,
            ))
    labels += auc_label if not zoomed else None,
    ax.set_ylabel('True Positive Rate')
    ax.set_xlabel('False Positive Rate')
    for reader in readers:
      sens = roc_row.get(f'{reader}_sens')
      spec = roc_row.get(f'{reader}_spec')
      sens_ci = (roc_row.get(f'{reader}_sens_ci_lower'), roc_row.get(f'{reader}_sens_ci_upper'))
      spec_ci = (roc_row.get(f'{reader}_spec_ci_lower'), roc_row.get(f'{reader}_spec_ci_upper'))

      reader_name = readers_props[reader]
      if not pd.isna(sens) and not pd.isna(spec):
        ax.add_line(lines.Line2D([1 - spec_ci[0], 1 - spec_ci[1]], [sens, sens],
                                zorder=-i,
                                linestyle='dotted',
                                c=color,
                                alpha=alpha,
        ))
        ax.add_line(lines.Line2D([1 - spec, 1 - spec], [sens_ci[0], sens_ci[1]],
                                zorder=-i,
                                linestyle='dotted',
                                c=color,
                                alpha=alpha,
        ))

        handles.extend(ax.plot(1 - spec, sens, alpha=alpha, zorder=100-i, **marker_kwawgs(reader, color, zoomed, marker_sizes[0], marker_sizes[1]), linestyle='None'))
        labels.append(reader_name)
    if zoomed:
      handles = [lines.Line2D([], [], linestyle='None', **marker_kwawgs(reader, 'grey', True, marker_sizes[0], marker_sizes[1]), label=reader)
                 for reader in readers]
      ax.legend(loc='lower right', handles=handles)
    else:
      if target == 'case':
        ax.legend(loc='lower right', title=f'{target.title()} Level', handles=handles, labels=labels)
      else:
        leg = plt_legend.Legend(ax, title=f'{target.title()} Level', handles=handles, labels=labels, loc='lower left')
        ax.add_artist(leg)





In [ ]:
@dataclasses.dataclass
class SideBySideSeries:
  subgroup_key: str
  subgroup_title: str
  prominent: bool = True
  colour: str = None

@dataclasses.dataclass
class SideBySideSection:
  series: list[SideBySideSeries]
  title: str = None
  target: str = 'case'
  colour: str = None
  readers: list[str] = dataclasses.field(default_factory=lambda: ['ai_read', 'human_first_read', 'human_second_read', 'human_consensus_read'])
  prefix: str = ''
  diff_reader: list[str] = 'ai_read'
  sens_only: bool = False
  title_pad: int = 0


def determine_test(noninferior_p, superior_p):
  if pd.isna(noninferior_p):
    return None, None, None
  elif noninferior_p < 0.05:
    assert not pd.isna(superior_p)
    if superior_p < 0.001:
      return 'superior', '<0.001', (noninferior_p, superior_p)
    elif superior_p < 0.05:
      return 'superior', f'={superior_p:.3f}', (noninferior_p, superior_p)
    elif noninferior_p < 0.001:
      return 'noninferior', '<0.001', (noninferior_p, superior_p)
    else:
      return 'noninferior', f'={noninferior_p:.3f}', (noninferior_p, superior_p)
  else:
    return None, None, (noninferior_p, None)

def plot_side_by_side(sections: list[SideBySideSection],
                      ranges_matched=False,
                      diff_range=None,
                      spec_lim=None,
                      spec_diff_lim=None,
                      ci_name='ci',
                      title=None,
                      minimum_n=None,
                      sens_only=False,
                      save=False,
                      inline_tests=False,
                      detailed_tests=True,
                      fig = None,
                      gridspec = None,
                      show_subgroup_titles = True,
                      nudges = {},
                      metrics_df=None,
                      add_counts=False,
                      marker_sizes=(6, 8),
                      tiny_fontsize=False,
                      ):
  tc = list(colors.TABLEAU_COLORS)
  readers_props = {
      'ai_read': (tc[0], 'AI Read'),
      'ai_read_strict':  (tc[0], 'AI Read †'),
      'ai_read_op1': (tc[0], 'AI Read'),
      'ai_read_op2': (tc[0], 'AI Read'),
      'human_first_read':  (tc[1], '1st Human Read'),
      'human_second_read':  (tc[2], '2nd Human Read'),
      'human_consensus_read': (tc[3], 'Human Consensus'),
  }
  title_fontsize = 'medium' if tiny_fontsize else 'large'
  section_title_fontsize = 'small' if tiny_fontsize else 'small'
  ytick_fontsize = 'x-small' if tiny_fontsize else 'medium'
  xtick_fontsize = 'small' if tiny_fontsize else 'medium'
  label_fontsize = 'small' if tiny_fontsize else 'medium'
  x_tick_pad = 10 if tiny_fontsize else 15
  section_linewidth = 0.6 if tiny_fontsize else 1.0
  inner_linewidth = 0.6 if tiny_fontsize else 1.0


  if metrics_df is None:
    raise ValueError('metrics_df is required')

  cols = 6 if detailed_tests else 4
  if fig:
    if gridspec:
      side_by_side_gridspec = gridspec.subgridspec(1, cols, hspace=0, wspace=0)
    else:
      side_by_side_gridspec = fig.add_gridspec(1, cols, hspace=0, wspace=0, bottom=0.2 if len(sections) <= 2 else None)
  else:
    fig = plt.figure(figsize=(20, 2 + 2 * len(sections)),
                     facecolor="white")
    fig.suptitle(title)
    side_by_side_gridspec = fig.add_gridspec(1, cols, hspace=0, wspace=0, bottom=0.2 if len(sections) <= 2 else None)

  ax1 = fig.add_subplot(side_by_side_gridspec[0, 0])
  ax2 = fig.add_subplot(side_by_side_gridspec[0, 1], sharey=ax1)
  ax3 = fig.add_subplot(side_by_side_gridspec[0, 2], sharey=ax1)
  ax4 = fig.add_subplot(side_by_side_gridspec[0, 3], sharey=ax1)
  if detailed_tests:
    ax5 = fig.add_subplot(side_by_side_gridspec[0, 4], sharey=ax1)
    ax6 = fig.add_subplot(side_by_side_gridspec[0, 5], sharey=ax1)
  else:
    ax5, ax6 = None, None

  ax1.set_title('Sensitivity', fontsize=title_fontsize)
  ax2.set_title('Specificity', fontsize=title_fontsize)
  ax3.set_title('Sensitivity difference vs AI', fontsize=title_fontsize)
  ax4.set_title('Specificity difference vs AI', fontsize=title_fontsize)
  if detailed_tests:
    ax5.set_title('Sensitivity', fontsize=title_fontsize)
    ax6.set_title('Specificity', fontsize=title_fontsize)
  ax2.tick_params(axis='y', length=0, labelleft=False)
  ax3.tick_params(axis='y', length=0, labelleft=False)
  ax4.tick_params(axis='y', length=0, labelleft=False)
  if detailed_tests:
    ax5.tick_params(axis='y', length=0, labelleft=False)
    ax6.tick_params(axis='y', length=0, labelleft=False)
  ax1.tick_params(axis='x', labelsize=xtick_fontsize)
  ax2.tick_params(axis='x', pad=x_tick_pad, labelsize=xtick_fontsize)
  ax3.tick_params(axis='x', labelsize=xtick_fontsize)
  ax4.tick_params(axis='x', pad=x_tick_pad, labelsize=xtick_fontsize)

  yticks, ylabels = [], []
  ax1.invert_yaxis()
  if detailed_tests:
    ax5.set_xticks([])
    ax6.set_xticks([])

  section_pad = 0.1
  reader_pad = 0.2
  series_pad = 0.15
  y = 0
  no_spec = []
  no_diff = []
  section_tops = []
  df_rows = []
  for section_index, section in enumerate(sections):
    section_tops.append(y)
    y += section.title_pad
    y += section_pad / 2
    has_spec = False
    has_diff = False
    for reader_index, reader in enumerate(section.readers):
      y += reader_pad / 2
      reader_colour, reader_name = readers_props[reader]
      for series_index, series in enumerate(section.series):
        row = pd.Series()
        df_rows.append(row)
        row['Section'] = section.title
        row['Subgroup'] = series.subgroup_title
        row['reader'] = reader_name
        colour = series.colour if series.colour else (section.colour if section.colour else reader_colour)
        marker_args = marker_kwawgs(reader, colour, markeredgewidth=0.6 if tiny_fontsize else 1)
        marker_args['markersize'] = marker_sizes[1] if series.prominent else marker_sizes[0]
        line_colour = colour
        y += series_pad / 2
        yticks.append(y)

        ylabels.append(f'{series.subgroup_title}')
        if series.subgroup_title is None:
          subgroup_col = metrics_df['Subgroup']
          subgroup_mask = (subgroup_col.isna() |
                           (subgroup_col.astype(str).str.strip() == '') |
                           (subgroup_col.astype(str).str.lower() == 'nan'))
        else:
          subgroup_mask = metrics_df['Subgroup'] == series.subgroup_title
        metrics_row = metrics_df[(metrics_df['Section'] == section.title) &
                                 subgroup_mask &
                                 (metrics_df['reader'] == reader_name)]
        if metrics_row.empty:
          y += series_pad / 2
          continue
        metrics_row = metrics_row.iloc[0]

        def render_metric(ax, metric, diff_stat=False, diff_ax=None, label_readers=False):
          has_metric = False
          metric_name = f'{metric}_diff' if diff_stat else metric
          col = metric_name
          if col not in metrics_df.columns and col.startswith('ai_read_strict_'):
            col = col.replace('ai_read_strict_', 'ai_read_')
          if col not in metrics_df.columns:
            return False
          val = metrics_row.get(col)
          val_ci = (metrics_row.get(f'{col} CI lower'), metrics_row.get(f'{col} CI upper'))
          if pd.notna(val) and pd.notna(val_ci[0]) and pd.notna(val_ci[1]):
            has_metric = True
            row[metric_name] = val
            row[f'{metric_name} CI lower'] = val_ci[0]
            row[f'{metric_name} CI upper'] = val_ci[1]
            ax.add_line(lines.Line2D([val_ci[0], val_ci[1]], [y, y], c=line_colour, linestyle='solid', linewidth=inner_linewidth))
            ax.plot(val, y, label=reader_name if section_index==0 and series_index ==0 and metric == 'sens' else None, **marker_args)
            if len(section.series) > 1:
              ax.axhline(y=y, c='grey', linestyle='dotted', linewidth=inner_linewidth, zorder=-2)
            if label_readers and series_index ==0:
              x_nudge = nudges.get(('reader_label_x', section_index, reader), 0)
              ax.text(val + x_nudge, y - 0.07, reader_name,
                      horizontalalignment='center',
                      verticalalignment='bottom',
                      fontsize=label_fontsize,
                      )
            if diff_stat:
              noninferior_p = metrics_row.get(f'{col} non-inferiority p-value')
              superior_p = metrics_row.get(f'{col} superiority p-value')
              test_type, p_value_text, (non_inferior_p_value, superior_p_value) = determine_test(noninferior_p, superior_p)
              row[f'{metric_name} non-inferiority p-value'] = noninferior_p
              row[f'{metric_name} superiority p-value'] = superior_p
              if inline_tests:
                marker = None
                if superior_p_value < 0.001:
                  marker = '+'
                elif superior_p_value < 0.01:
                  marker = '^'
                elif non_inferior_p_value < 0.001:
                  marker = '*'
                if marker:
                  ax.plot(val, y - 0.15, marker=marker, color='black',)
              if detailed_tests:
                if test_type:
                  diff_text =  f'{test_type} [p{p_value_text}]'
                  diff_ax.text(diff_ax.get_xlim()[0] + 0.1, y, f'{section.diff_reader} {diff_text}')
          return has_metric

        render_metric(ax1, 'sens', label_readers=True)
        has_diff = render_metric(ax3, f'{section.diff_reader}_sens', diff_stat=True, diff_ax=ax5) or has_diff
        if not sens_only and not section.sens_only:
          has_spec = render_metric(ax2, 'spec') or has_spec
          render_metric(ax4, f'{section.diff_reader}_spec', diff_stat=True, diff_ax=ax6)
        y += series_pad / 2
      y += reader_pad / 2
    if not has_spec:
      no_spec.append(section_index)
    if not has_diff:
      no_diff.append(section_index)
    y += section_pad / 2
  ax1.set_ylim(y, 0)
  ax1.set_yticks(yticks if show_subgroup_titles else [])
  ax1.set_yticklabels(ylabels if show_subgroup_titles else [], fontsize=ytick_fontsize)

  if ranges_matched:
    size = max((ax1.get_xlim()[1] - ax1.get_xlim()[0]),
               (ax2.get_xlim()[1] - ax2.get_xlim()[0]))
    size = min(size, 1)
    ax1mid = ax1.get_xlim()[0] + (ax1.get_xlim()[1] - ax1.get_xlim()[0]) / 2
    ax2mid = ax2.get_xlim()[0] + (ax2.get_xlim()[1] - ax2.get_xlim()[0]) / 2

    ax1lim = (ax1mid - size/2, ax1mid + size/2)
    ax2lim = (ax2mid - size/2, ax2mid + size/2)
    if ax1lim[0] < 0:
      ax1lim = (0, size)
    if ax2lim[0] < 0:
      ax2lim = (0, size)
    if ax1lim[1] > 1:
      ax1lim = (1- size, 1)
    if ax2lim[1] > 1:
      ax2lim = (1- size, 1)
    ax1.set_xlim(ax1lim)
    ax2.set_xlim(ax2lim)
    if sens_only:
      diff_lim = max(abs(ax3.get_xlim()[0]),
                     abs(ax3.get_xlim()[1]))
    else:
      diff_lim = max(abs(ax3.get_xlim()[0]),
                     abs(ax3.get_xlim()[1]),
                     abs(ax4.get_xlim()[0]),
                     abs(ax4.get_xlim()[1]))
    ax3.set_xlim((-diff_lim, diff_lim))
    ax4.set_xlim((-diff_lim, diff_lim))
  else:
    ax3lim = max(0.1, abs(ax3.get_xlim()[0]), abs(ax3.get_xlim()[1]))
    ax3.set_xlim(-ax3lim, ax3lim)
    ax4lim = max(0.1, abs(ax4.get_xlim()[0]), abs(ax4.get_xlim()[1]))
    ax4.set_xlim(-ax4lim, ax4lim)
  if 'sens_xlim_max' in nudges:
    ax1.set_xlim(ax1.get_xlim()[0], nudges['sens_xlim_max'])

  if diff_range:
    ax3.set_xlim(diff_range)
    ax4.set_xlim(diff_range)

  if spec_lim:
    ax2.set_xlim(spec_lim)
  if spec_diff_lim:
    ax4.set_xlim(spec_diff_lim)

  non_inferiority_margin = 0.05
  for i, ax in enumerate([ax3, ax4]):
    ax.add_patch(patches.Rectangle(
        (0, ax.get_ylim()[0]),
        -non_inferiority_margin,
        (ax.get_ylim()[1] - ax.get_ylim()[0]),
        linewidth=1, color='grey', zorder=-1, alpha=0.25,
        label='Non-Inferiority Margin' if i == 0 else None))
    if not tiny_fontsize:
      ax.add_line(lines.Line2D([0, 0], [ax.get_ylim()], c='black', linestyle='dotted', linewidth=1, zorder=-2))
      ax.add_line(lines.Line2D([-non_inferiority_margin, -non_inferiority_margin], [ax.get_ylim()], c='black', linestyle='dotted', linewidth=1, zorder=-2))
    ax.text(0.01, (section_pad + reader_pad + series_pad) / 2 - 0.05, f"AI Read better", verticalalignment='bottom', fontsize=label_fontsize)
    ax.text(-0.01, (section_pad + reader_pad + series_pad) / 2 - 0.05, f"AI Read worse", verticalalignment='bottom', horizontalalignment='right', fontsize=label_fontsize)
    ax.annotate("", xy=(0, (section_pad + reader_pad + series_pad) / 2), xytext=(-50, 0), textcoords='offset points', arrowprops=dict(arrowstyle="<-"))
    ax.annotate("", xy=(0, (section_pad + reader_pad + series_pad) / 2), xytext=(50, 0), textcoords='offset points', arrowprops=dict(arrowstyle="<-"))

  if len(section_tops) > 1:
    ax3.text(-non_inferiority_margin, nudges.get('ni_margin_y_pos', 0.5) * section_tops[1], nudges.get('ni_margin_label', 'Non-Inferiority\nMargin'), rotation=90, horizontalalignment='center', verticalalignment='center')

  section_tops.append(ax1.get_ylim()[0])
  for ax, indexes in [(ax2, no_spec), (ax3, no_diff), (ax4, no_diff)]:
    for i in indexes:
      ax.add_patch(patches.Rectangle(
        (ax.get_xlim()[0], section_tops[i]),
        (ax.get_xlim()[1] - ax.get_xlim()[0]),
        -(section_tops[i] - section_tops[i + 1]),
        linewidth=1, color='whitesmoke', zorder=1, alpha=1,))

  for i, (section, section_top) in enumerate(zip(sections, section_tops)):
    if section.title:
      lim = ax1.get_xlim()
      if tiny_fontsize:
        bbox_dict = dict(facecolor='white', alpha=0.5, pad=0.5, linewidth=0.5)
      else:
        bbox_dict = dict(facecolor='white', alpha=0.5)
      ax1.text(lim[0] + 0.05 * (lim[1] - lim[0]),  # 0.05
               section_top + section_pad,
               section.title, bbox=bbox_dict, verticalalignment='top',
               fontsize=section_title_fontsize,
               )

    for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
      if ax is None:
        continue
      if i > 0:
        ax.add_line(lines.Line2D([ax.get_xlim()], [section_top, section_top], c='black', linestyle='dashed', linewidth=section_linewidth))

  if sens_only:
    ax2.remove()
    ax4.remove()
    ax6.remove()







In [ ]:
from enum import Enum, auto

class LegendType(Enum):
  TEXT_ONLY = auto()
  TEXT_AND_COUNT = auto()
  TEXT_AND_COUNT_SHORT = auto()
  COUNT_ONLY = auto()
  COUNT_ONLY_SHORT = auto()
  NONE = auto()

@dataclasses.dataclass
class CDR_RR_Series:
  subgroup: str
  subgroup_label: str
  colour: str
  prominent: bool = False
  render: bool = True
  legend: LegendType = LegendType.TEXT_ONLY
  curve: tuple[np.array, np.array] = None

def plot_cdr_rr(ax,
                series: list[CDR_RR_Series],
                target = 'case',
                readers = ['ai_read', 'human_first_read', 'human_second_read', 'human_consensus_read'],
                averages = ['2016'],
                rr_lim = None, #[2, 10],
                cdr_lim = None, #[5, 12],
                legend2_loc = 'upper right',
                cdr_df = None,
                grouping_metrics_df = None,
                marker_sizes = (8, 10)
                ):
  readers_props = {
        'ai_read': ('AI Read'),
        'ai_read_strict':  ('AI Read'),
        'ai_read_op1': ('AI Read'),
        'ai_read_op2': ('AI Read'),
        'human_first_read':  ('1st Human Read'),
        'human_second_read':  ('2nd Human Read'),
        'human_consensus_read': ('Human Consensus'),
  }

  if cdr_df is None:
    raise ValueError('cdr_df is required')

  site_handles = []

  cdr_rows = cdr_df.set_index('Group')
  rr_cols = sorted([c for c in cdr_df.columns if c.startswith('RR_')], key=lambda c: int(c.split('_')[1]))
  cdr_cols = sorted([c for c in cdr_df.columns if c.startswith('CDR_')], key=lambda c: int(c.split('_')[1]))

  for i, s in enumerate(series):
    match s.legend:
      case LegendType.TEXT_ONLY:
        site_handles.append(patches.Patch(color=s.colour, label=f'{s.subgroup_label}'))
      case LegendType.TEXT_AND_COUNT:
        if grouping_metrics_df is not None:
          count = grouping_metrics_df.loc[f'{s.subgroup}:{target}'].n
          site_handles.append(patches.Patch(color=s.colour, label=f'{s.subgroup_label} (count={count})'))
      case LegendType.TEXT_AND_COUNT_SHORT:
        if grouping_metrics_df is not None:
          count = grouping_metrics_df.loc[f'{s.subgroup}:{target}'].n
          site_handles.append(patches.Patch(color=s.colour, label=f'{s.subgroup_label} (n={count})'))
      case LegendType.COUNT_ONLY:
        if grouping_metrics_df is not None:
          count = grouping_metrics_df.loc[f'{s.subgroup}:{target}'].n
          site_handles.append(patches.Patch(color=s.colour, label=f'count={count}'))
      case LegendType.COUNT_ONLY_SHORT:
        if grouping_metrics_df is not None:
          count = grouping_metrics_df.loc[f'{s.subgroup}:{target}'].n
          site_handles.append(patches.Patch(color=s.colour, label=f'n={count}'))
      case LegendType.NONE:
        pass
    if not s.render:
      continue

    if s.subgroup_label not in cdr_rows.index:
      continue
    data_row = cdr_rows.loc[s.subgroup_label]
    if isinstance(data_row, pd.DataFrame):
      data_row = data_row.iloc[0]

    for reader in readers:
      rr = data_row.get(f'{reader} RR')
      cdr = data_row.get(f'{reader} CDR')
      rr_ci = (data_row.get(f'{reader} RR CI lower'), data_row.get(f'{reader} RR CI upper'))
      cdr_ci = (data_row.get(f'{reader} CDR CI lower'), data_row.get(f'{reader} CDR CI upper'))
      if pd.isna(rr) or pd.isna(cdr):
        continue

      marker_args = marker_kwawgs(reader, s.colour, markersize=marker_sizes[0], zoomed_markersize=marker_sizes[1])
      marker_args['markersize'] = marker_sizes[1] if s.prominent else marker_sizes[0]

      low_prom_alpha = 0.35
      line_alpha = 1 if s.prominent else low_prom_alpha
      if pd.notna(rr_ci[0]) and pd.notna(rr_ci[1]):
        ax.add_line(lines.Line2D([rr_ci[0], rr_ci[1]], [cdr, cdr],
                                zorder=(10 - i),
                                linestyle='dotted',
                                c=s.colour,
                                alpha=line_alpha,
        ))
      if pd.notna(cdr_ci[0]) and pd.notna(cdr_ci[1]):
        ax.add_line(lines.Line2D([rr, rr], [cdr_ci[0], cdr_ci[1]],
                                zorder=(10 - i),
                                linestyle='dotted',
                                c=s.colour,
                                alpha=line_alpha,
        ))
      ax.plot(rr, cdr, **marker_args, linestyle='None', alpha=1 if s.prominent else low_prom_alpha, zorder=(100 - 10 * i))

    ax.add_artist(plt_legend.Legend(
        ax,
        handles=[lines.Line2D([], [], linestyle='None', **marker_kwawgs(reader, 'grey', True, marker_sizes[0], marker_sizes[1]), label=reader) for reader in readers],
        labels=[readers_props[reader] for reader in readers],
        loc=legend2_loc, alignment='left'))

  legend2 = plt_legend.Legend(ax, handles=site_handles, labels=[h.get_label() for h in site_handles], loc='upper left', alignment='left')
  ax.add_artist(legend2)

  ax.set_xlim(rr_lim)
  ax.set_ylim(cdr_lim)
  ax.set_xlim(ax.get_xlim()) # Freeze lims as is, if they aren't set
  ax.set_ylim(ax.get_ylim())

  for i, s in enumerate(series):
    if s.subgroup_label not in cdr_rows.index:
      continue
    data_row = cdr_rows.loc[s.subgroup_label]
    if rr_cols and cdr_cols:
      rr_curve = data_row[rr_cols].to_numpy()
      cdr_curve = data_row[cdr_cols].to_numpy()
      ax.plot(rr_curve, cdr_curve,
          c=s.colour,
          alpha=1 if s.prominent else 0.35,
          )

  ax.set_xlabel('Recall rate (%)')
  ax.set_ylabel('Cancer detection per 1,000')
  if averages:
    ax.legend(loc='lower right')



In [ ]:
tab_colours = list(colors.TABLEAU_COLORS)
case_colour = tab_colours[0]
breast_colour = tab_colours[1]
lesion_colour = tab_colours[2]


def figure_2_roc(outer_ax):
  zoom_xy = ([0.0, .19], [0.375, .575])
  inner_height = .64
  inner_top_left = 0.025
  inner_width = (zoom_xy[0][1] - zoom_xy[0][0]) / (zoom_xy[1][1] - zoom_xy[1][0]) * inner_height
  inner_ax = outer_ax.inset_axes([1 - inner_width - inner_top_left,
                                  1 - inner_height - inner_top_left,
                                  inner_width,
                                  inner_height])
  inner_ax.set_facecolor((1, 1, 1, 0.85))

  plot_readers_and_auc_pair(outer_ax, inner_ax, subgroups=['ALL'],
                            label_with_target=True, target='case', zoom_xy=zoom_xy, marker_sizes=(5, 7),
                            readers = ['ai_read', 'human_first_read', 'human_second_read', 'human_consensus_read'],
                            roc_df=all_sheets['Figure 2b']
                            )
  plot_readers_and_auc_pair(outer_ax, inner_ax, subgroups=['ALL'],
                            label_with_target=True, target='breast', subgroup_colors=['orange'], zoom_xy=zoom_xy, marker_sizes=(5, 7),
                            readers = ['ai_read', 'human_first_read', 'human_second_read', 'human_consensus_read'],
                            roc_df=all_sheets['Figure 2b']
                            )
  inner_ax.set_xlabel(None)
  inner_ax.set_ylabel(None)
  inner_ax.yaxis.set_major_locator(plt.MaxNLocator(4))

def figure_2_cdr_rr(ax):
  cs = list(colors.TABLEAU_COLORS)
  site_cols = ['#009E73','#56B4E9','#E69F00','#F0E442','#D55E00']

  series = [
      CDR_RR_Series('ALL', 'All Screening Services', cs[0], prominent=True),
  ] + [
      CDR_RR_Series(None, f'Screening Service {i+1}', site_cols[i]) for i in range(5)
  ]

  plot_cdr_rr(ax,
              series=series,
              averages = [],
              legend2_loc = 'lower right',
              cdr_lim = (None, 12.5),
              cdr_df=all_sheets['Figure 2c'],
              marker_sizes = (4, 6),
              )

def figure_2_side_by_side(fig, gs):
  title_pad = .3
  sections = [
      SideBySideSection(series=[SideBySideSeries(None, 'All Cases')], title='Case Level', colour=case_colour,
                    readers=['human_first_read', 'human_second_read', 'human_consensus_read', 'ai_read'], title_pad=title_pad),
      SideBySideSection(series=[SideBySideSeries(None, 'All Cases')], title='Breast Level', colour=breast_colour, target='breast',
                    readers=['human_first_read', 'human_second_read', 'human_consensus_read', 'ai_read'], title_pad=title_pad),
      SideBySideSection(series=[SideBySideSeries(None, 'All Cases')], title='Lesion Level', colour=lesion_colour, target='breast',
                        diff_reader='ai_read_strict', prefix='Strict:', readers=['ai_read_strict'], title_pad=title_pad),
  ]
  plot_side_by_side(sections=sections,
                    ranges_matched = True,
                    inline_tests = True,
                    fig = fig,
                    gridspec = gs,
                    nudges = {('reader_label_x', 1, 'human_first_read'): 0.025},
                    detailed_tests = False,
                    show_subgroup_titles = False,
                    metrics_df=all_sheets['Figure 2a'],
                    marker_sizes = (4, 6),
  )

def figure_2_recall(ax1):
  cs = list(colors.TABLEAU_COLORS)
  series = [
      CDR_RR_Series('ALL', 'All Screens', cs[0], prominent=True),
  ] + [
      CDR_RR_Series(None, subgroup_title, cs[i + 1], prominent=True)
      for i, subgroup_title in enumerate(display_subgroups['By Screen Type'])
  ]

  plot_cdr_rr(ax1,
              series=series,
              cdr_lim = [6.5, 12],
              rr_lim = [0, 15],
              averages = [],
              cdr_df=all_sheets['Figure 2d'],
              marker_sizes = (4, 6),
              legend2_loc = 'lower right',
                )

def figure2(figsize=(12, 12), dpi=72):
  fig = plt.figure(figsize=figsize)
  pad = 0.04
  space = 0.15
  gs2x2 = fig.add_gridspec(3, 2, hspace=space, wspace=space, top=1-pad, left=pad + 0.01, bottom=pad, right=1-pad, height_ratios=[2, 2, 1])
  figure_2_side_by_side(fig, gs2x2[0,:])

  axROC = fig.add_subplot(gs2x2[1, 0])
  figure_2_roc(axROC)
  axb2 = fig.add_subplot(gs2x2[1, 1])
  figure_2_cdr_rr(axb2)

  axRecall = fig.add_subplot(gs2x2[2, :])
  figure_2_recall(axRecall)
  text_args = dict(size='x-large',
                   horizontalalignment='left',
                   verticalalignment='center')
  fig.text(0.05, 0.97, 'a)', **text_args)
  fig.text(0.05, 0.595, 'b)', **text_args)
  fig.text(0.535, 0.595, 'c)', **text_args)
  fig.text(0.05, 0.22, 'd)', **text_args)
  return fig

plt.rcParams['font.size'] = 6.0
fig2 = figure2(figsize=(to_inch(180), to_inch(225)))
plt.rcParams['font.size'] = 10.0


In [ ]:
plt.rcParams['font.size'] = 6.0
tab_colours = list(colors.TABLEAU_COLORS)
all_cases_colour = tab_colours[0]
subgroup_colour = tab_colours[1]


sections = [
    SideBySideSection(series=[SideBySideSeries(None, 'All Cases')] +
                              [SideBySideSeries(None, subgroup, prominent=False, colour=subgroup_colour)
                              for subgroup in display_subgroups[subgrouping]],
                      colour=all_cases_colour,
                      sens_only=sens_only,
                      title=subgrouping,
                      title_pad=0.33,
                      readers=['ai_read', 'human_first_read'],
                      )
    for (subgrouping, sens_only) in [
                        ('By Screening Service', False),
                        ('By Screen Type', False),
                        ('By Cancer Time-Point', True),
                        ('By Age', False),
                        ('By Index of Multiple Deprivation', False),
                        ('By Ethnicity', False),
                        ('By Breast Density', False),
                        ('By Lesion Type', True),
                        ]
]
fig3 = plt.figure(figsize=(to_inch(180), to_inch(225)))
plot_side_by_side(sections=sections,
                  ranges_matched = True,
                  detailed_tests = False,
                  fig = fig3,
                  spec_lim = (0.75, 1),
                  spec_diff_lim = (-0.1, 0.1),
                  metrics_df=all_sheets['Figure 3'],
                  marker_sizes = (3, 4),
                  tiny_fontsize=True
)
fig3.tight_layout()
plt.rcParams['font.size'] = 10.0


In [ ]:
def weekly_trends(axes, legend_ax, include_label = None, sites_in_legend=True, weekly_df=None, markersize=6):
  if weekly_df is None:
    raise ValueError('weekly_df is required')
  cs = list(colors.TABLEAU_COLORS)[3:]

  df = weekly_df.set_index('Unnamed: 0')
  df.columns = pd.to_datetime(df.columns)
  df = df.apply(pd.to_numeric, errors='coerce')
  dates = df.columns

  def row_series(name):
    if name not in df.index:
      return None
    return pd.Series(df.loc[name].values, index=dates)

  ax = axes[0]
  for label, colour, orig_op_index in [('Screening Service 1', cs[0], 3),
                      ('Screening Service 2', cs[1], 2)]:
    if label != include_label:
      continue
    ai_rr = row_series(f'{label} AI Read RR')
    ai_rr_ci_lower = row_series(f'{label} AI Read RR CI lower')
    ai_rr_ci_upper = row_series(f'{label} AI Read RR CI upper')
    ai_rr_op = row_series(f'{label} AI Read (Original OP) RR')
    human_rr = row_series(f'{label} Human First Read RR')
    human_rr_ci_lower = row_series(f'{label} Human First Read RR CI lower')
    human_rr_ci_upper = row_series(f'{label} Human First Read RR CI upper')

    if ai_rr is None or human_rr is None:
      continue

    ax.plot(dates, ai_rr, **marker_kwawgs('ai_read', colour, markersize=markersize), label=f'{label} - AI Read')
    if ai_rr_op is not None:
      ax.plot(dates[orig_op_index:], ai_rr_op.iloc[orig_op_index:], **marker_kwawgs('ai_read', colour, markersize=markersize), linestyle="dotted", label=f'{label} - AI Read (Original OP)')
    ax.plot(dates, human_rr, **marker_kwawgs('human_first_read', colour, markersize=markersize), linestyle="dashed", label=f'{label} - Human First Read')
    if ai_rr_ci_lower is not None and ai_rr_ci_upper is not None:
      ax.fill_between(dates, ai_rr_ci_lower, ai_rr_ci_upper, color=colour, alpha=.05)
    if human_rr_ci_lower is not None and human_rr_ci_upper is not None:
      ax.fill_between(dates, human_rr_ci_lower, human_rr_ci_upper, color=colour, alpha=.05)
    ax.set_ylim(0, 17)
    if not legend_ax:
      ax.legend(ncols=2)
    else:
        legend_ax.add_artist(plt_legend.Legend(
          legend_ax,
          handles=[lines.Line2D([], [], **marker_kwawgs('ai_read', 'grey')),
                   lines.Line2D([], [], **marker_kwawgs('ai_read', 'grey'), linestyle='dotted'),
                   lines.Line2D([], [], **marker_kwawgs('human_first_read', 'grey'), linestyle='dashed'),
                   ],
          labels=['AI Read', 'AI Read (Initial OP cont.)', 'Human First Read'],
          handlelength=4,
          fontsize='small',
          loc='center right' if sites_in_legend else 'upper center',
          ncols=1 if sites_in_legend else 3,
          alignment='left',
          borderaxespad=0.01
          ))
    if not sites_in_legend:
      ax.set_title(label)

    ax.set_ylabel('Weekly Recall Rate (%)')

  ax = axes[1]
  for label, colour in [('Screening Service 1', cs[0]),
                      ('Screening Service 2', cs[1])]:
    if label != include_label:
      continue
    ai_cdr = row_series(f'{label} AI Read CDR')
    ai_cdr_ci_lower = row_series(f'{label} AI Read CDR CI lower')
    ai_cdr_ci_upper = row_series(f'{label} AI Read CDR CI upper')
    human_cdr = row_series(f'{label} Human First Read CDR')
    human_cdr_ci_lower = row_series(f'{label} Human First Read CDR CI lower')
    human_cdr_ci_upper = row_series(f'{label} Human First Read CDR CI upper')

    if ai_cdr is None or human_cdr is None:
      continue

    ax.plot(dates, ai_cdr + 0.2, **marker_kwawgs('ai_read', colour, markersize=markersize), label=f'{label} - AI Read')
    ax.plot(dates, human_cdr, **marker_kwawgs('human_first_read', colour, markersize=markersize), linestyle="dashed", label=f'{label} - Human First Read')
    if ai_cdr_ci_lower is not None and ai_cdr_ci_upper is not None:
      ax.fill_between(dates, ai_cdr_ci_lower, ai_cdr_ci_upper, color=colour, alpha=.05)
    if human_cdr_ci_lower is not None and human_cdr_ci_upper is not None:
      ax.fill_between(dates, human_cdr_ci_lower, human_cdr_ci_upper, color=colour, alpha=.05)
  ax.set_ylabel('Cancer Detection Rate\nper 1,000 women')
  ax.set_ylim(0, 16.5)
  if not legend_ax:
    ax.legend(ncols=2)

  ax = axes[2]
  legend_handles = []
  legend_labels = []
  for label, colour in [('Screening Service 1', cs[0]),
                      ('Screening Service 2', cs[1])]:
    if label != include_label:
      continue
    num_cases = row_series(f'{label} Number of Cases')
    if num_cases is None:
      continue
    ax.plot(dates, num_cases, label=label, marker="o", c=colour)
    legend_handles.append(patches.Patch(color=colour))
    legend_labels.append(label)
  ax.set_ylim(0, 1100)
  ax.set_yticks([0, 300, 600, 900])
  ax.set_xlabel('Study week')
  ax.set_ylabel('Cases per Week')
  ax.set_xticks(dates)
  ax.set_xticklabels(range(1, len(dates) + 1))
  if not legend_ax:
    ax.legend(loc="lower right")
  elif sites_in_legend:
      legend_ax.add_artist(plt_legend.Legend(
        legend_ax,
        handles=legend_handles,
        labels=legend_labels,
        loc='center left', alignment='left'))
  for ax in axes:
    ax.axvline(x=datetime(2023, 12, 22), linestyle='dashed', c='grey')
  axes[2].text(datetime(2023, 12, 21), 10, 'Initial OP', horizontalalignment='right', verticalalignment='bottom', size='small')
  axes[2].text(datetime(2023, 12, 23), 10, 'Adjusted OP', horizontalalignment='left', verticalalignment='bottom', size='small')

def figure_4_weekly(fig, gridspec, weekly_df, markersize=6):
  weekly_gridspec = gridspec.subgridspec(10, 2, hspace=0, wspace=.05)
  axLegend = fig.add_subplot(weekly_gridspec[0, :])
  axLegend.axis('off')
  ax1 = fig.add_subplot(weekly_gridspec[1:4, 0])
  ax2 = fig.add_subplot(weekly_gridspec[4:7, 0])
  ax3 = fig.add_subplot(weekly_gridspec[7:10, 0])
  weekly_trends([ax1, ax2, ax3], axLegend, include_label='Screening Service 1', sites_in_legend=False, weekly_df=weekly_df, markersize=markersize)
  ax4 = fig.add_subplot(weekly_gridspec[1:4, 1])
  ax5 = fig.add_subplot(weekly_gridspec[4:7, 1])
  ax6 = fig.add_subplot(weekly_gridspec[7:10, 1])
  weekly_trends([ax4, ax5, ax6], axLegend, include_label='Screening Service 2', sites_in_legend=False, weekly_df=weekly_df, markersize=markersize)

  ax1.set_xticklabels([])
  ax2.set_xticklabels([])
  ax4.set_xticklabels([])
  ax5.set_xticklabels([])
  ax4.yaxis.tick_right()
  ax5.yaxis.tick_right()
  ax6.yaxis.tick_right()
  ax4.set_ylabel(None)
  ax5.set_ylabel(None)
  ax6.set_ylabel(None)




In [ ]:
tab_colours = list(colors.TABLEAU_COLORS)
prospective_colour = tab_colours[0]
tranche1_colour = tab_colours[1]
tranche2_colour = tab_colours[2]


def figure_4_cdr_rr(ax):
  cs = list(colors.TABLEAU_COLORS)
  series = [
      CDR_RR_Series(None, 'Retrospective', cs[0], prominent=True),
      CDR_RR_Series(None, 'Prospective - Initial OP', cs[1], prominent=True),
      CDR_RR_Series(None, 'Prospective - Adjusted OP', cs[2], prominent=True),
  ]

  cdr_lim =[3, 13]
  rr_lim = [2.5, 14]

  plot_cdr_rr(ax,
              series=series,
              cdr_lim = cdr_lim,
              rr_lim = rr_lim,
              averages = [],
              cdr_df = all_sheets['Figure 4b'],
              marker_sizes = (4, 6),
                  )

def figure_4_side_by_side(fig, gs):
  title_pad=0.35
  plot_side_by_side(sections=[
                        SideBySideSection(series=[SideBySideSeries(None, None)], title='Retrospective',
                                          readers=['human_first_read', 'human_second_read', 'human_consensus_read', 'ai_read'],
                                          colour=prospective_colour, title_pad=title_pad),
                        SideBySideSection(series=[SideBySideSeries(None, 'OP1')],
                                          title='Prospective - Initial OP',
                                          readers=['human_first_read', 'human_second_read', 'human_consensus_read', 'ai_read_op1'],
                                          diff_reader='ai_read_op1',
                                          colour=tranche1_colour, title_pad=title_pad),
                        SideBySideSection(series=[SideBySideSeries(None, 'OP2')],
                                          title='Prospective - Adjusted OP',
                                          readers=['human_first_read', 'human_second_read', 'human_consensus_read', 'ai_read_op2'],
                                          diff_reader='ai_read_op2',
                                          colour=tranche2_colour, title_pad=title_pad),
                    ],
                    ranges_matched = True,
                    show_subgroup_titles = False,
                    nudges = {('reader_label_x', 0, 'human_second_read'): -0.04,
                              ('reader_label_x', 0, 'human_consensus_read'): -0.07,
                              ('reader_label_x', 1, 'human_second_read'): -0.065,
                              ('reader_label_x', 1, 'human_consensus_read'): -0.07,
                              ('reader_label_x', 2, 'human_second_read'): -0.045,
                              ('reader_label_x', 2, 'human_consensus_read'): -0.07,
                              'sens_xlim_max': 1.01,
                              'ni_margin_label': 'NI\nMargin',
                              'ni_margin_y_pos': 0.28,
                                },
                    fig = fig,
                    gridspec = gs,
                    detailed_tests = False,
                    metrics_df = all_sheets['Figure 4a'],
                    add_counts = True,
                    marker_sizes = (3, 5),
  )


def figure4(figsize=(12, 12), dpi=72):
  fig = plt.figure(figsize=figsize)
  pad = 0.04
  space = 0.15
  gs2x2 = fig.add_gridspec(2, 2, hspace=space, wspace=space, top=1-pad, left=pad + 0.02, bottom=pad, right=1-pad,)
  figure_4_side_by_side(fig, gs2x2[0,:])

  figure_4_weekly(fig, gs2x2[1, 1], all_sheets['Figure 4c'], markersize=4)

  axb2 = fig.add_subplot(gs2x2[1, 0])
  figure_4_cdr_rr(axb2)

  text_args = dict(size='x-large',
                   horizontalalignment='left',
                   verticalalignment='center')
  fig.text(0.05, 0.97, 'a)', **text_args)
  fig.text(0.05, 0.48, 'b)', **text_args)
  fig.text(0.535, 0.48, 'c)', **text_args)
  return fig

plt.rcParams['font.size'] = 6.0
fig4 = figure4(figsize=(to_inch(180), to_inch(180)))
plt.rcParams['font.size'] = 10.0


In [ ]:
def render_dpi(fig, dpi=300, format='png'):
  buf = io.BytesIO()
  fig.savefig(buf, format=format, dpi=dpi, bbox_inches='tight')
  buf.seek(0)
  plt.close(fig)
  img = Image.open(buf)
  return img

In [ ]:
render_dpi(fig2)

In [ ]:
render_dpi(fig3)

In [ ]:
render_dpi(fig4)